# 手順5a：温度シフト構造の見直し（新データ）

CLAUDE.md §5「見直し中：シフト構造」の確認。当初の構造（時間–温度換算則を、設定温度 Tset で組み込む）が、新しいデータで成り立つかを調べ、代わりになる構造を比べる。

**なぜ見直すのか**：当初の構造では、横方向（せん断速度の軸）と縦方向（粘度の軸）を**同じ活性化エネルギー E** でずらす。ところが新データの見かけの流動曲線は、傾き s ≈ −1.6（せん断速度が上がると、せん断応力が下がる）。そのため、同じせん断速度での粘度の差は (1 + s)·Δln aT になり、E > 0 にすると「低温ほど粘度が低い」予測になってしまう。データは「低温ほど粘度が高い」（190 ℃ は 230 ℃ の 1.09〜1.21 倍）。試算では、最適な E が 0 付近か負になった（−5〜−2.5 kJ/mol）。

**比べる構造**（f は GP。入力は u = ln γ̇ ＋ ln aT_h、w = ln Fo = −(ln γ̇ ＋ 3 ln H) ＋ 定数。シフトは ln a(T; E) = (E/R)·(1/T − 1/Tref)、Tref = 483.15 K（210 ℃））

| 記号 | 構造 | 式 | 推定するもの |
|---|---|---|---|
| H | 当初（§5）：横も縦も同じ E | ln η = ln a(E) + f(ln γ̇ + ln a(E), ln Fo) | E |
| A | 縦方向だけシフト | ln η = ln a(E_v) + f(ln γ̇, ln Fo) | E_v |
| Hz | 横方向だけシフト | ln η = f(ln γ̇ + ln a(E_h), ln Fo) | E_h |
| B | 横と縦を別々 | ln η = ln a(E_v) + f(ln γ̇ + ln a(E_h), ln Fo) | E_h, E_v |
| N | 温度なし（H の E = 0） | ln η = f(ln γ̇, ln Fo) | ― |
| H40 | H で E = 40 kJ/mol に固定（文献値） | ― | ― |
| F | 参考：Tff でシフト（H と同じ式で、T に Tff を使う） | ― | E |
| C | 比較用：構造なし。Tset を入力に並べた通常の GPR（R1） | ln η = f(ln γ̇, ln H, Tset) | ― |

**推定（プロファイル尤度）**：E を格子で振り、各 E で入力を (u, ln Fo)、出力を ln η − ln a(E_v) に変換して GPR（ARD Matern 5/2 ＋ 線形カーネル ＋ ノイズ）を学習し、対数周辺尤度が最大になる E を採用する。出力の自動標準化は使わない（E ごとに尺度が変わり、尤度を比べられなくなるため）。入力の標準化も、E に依存しない定数（E = 0 のときの学習データの平均・標準偏差）を使う。**尤度の比較は条件平均**（条件ごとに、ショットの ln η と ln γ̇ を平均した点）で行う。ショットごとの γ̇ を入力にすると、E ≠ 0 のとき設定温度が入力の上で少し分かれ、GP がそれを温度の目印に使えてしまうため（CLAUDE.md §5）。

**予測の評価**：尤度で決めた E を使って、ショット単位の GPR を学習し、S_V（5data → 3data）と S_T（190・230 ℃ → 210 ℃）で評価する。入力の γ̇ は、条件平均（cm：成形条件から見た予測）と、ショットごと（shot）の両方。

- 「A」「B」の E_v が 7 kJ/mol 程度なら、190 ℃ と 230 ℃ の粘度の比は約 1.15 倍（exp(0.0206·E_v)）。

**本命と対案（ユーザー決定、2026-10-12）**：本命は **B**（横・縦を別々）、対案は **A**（縦だけ）。後で自分で実行して比べられるように、この 2 つを並べて見る節（「8. 本命と対案の比較」）を設けた。比べる構造は、設定エリアの `MAIN_STRUCT`・`ALT_STRUCT` で変えられる（H・A・Hz・B・N・H40・F・C のどれでもよい）。比較の基準は C（構造なしの通常の GPR）。

**それぞれの形で、何が表せるか（なぜ採用したか）**

| 記号 | この形で表せること | 採用したときの期待 |
|---|---|---|
| H | 温度を変えると、流動曲線が形を変えずに、横（せん断速度）と縦（粘度）に、同じ割合でずれる。1 本の基準の曲線（マスターカーブ）から、どの温度の曲線も作れる。 | 当初の構造（CLAUDE.md §5）。温度ごとのデータを 1 本にまとめられ、E を文献値（PP は 40 kJ/mol 前後）と比べられる、と期待した。 |
| A | 曲線の形は温度で変わらず、全体が一定の倍率だけ上下に動く。同じ γ̇ で、低温ほど粘度が一定の倍率だけ高い。 | 実測の「190 ℃ は 230 ℃ の 1.1〜1.2 倍」を、パラメータ 1 個（E_v）でいちばん簡単に表せる、と期待した。 |
| Hz | 曲線の形は温度で変わらず、全体が横にだけ動く。同じ粘度になる γ̇ が、温度で変わる。 | 横のずれだけで温度の効果を表せるかを確かめる対照。 |
| B | A の「一定の倍率の上下」に、横のずれを足す。曲線の傾きは γ̇ の場所で違うので、横にずらすと、粘度の差が場所ごとに変わる。 | 実測の「厚いほど（γ̇ が小さいほど）温度の効果が大きい」傾向を、A のわかりやすさ（一定の倍率が土台）を残したまま表せる、と期待した。 |
| N | 温度を無視する（E = 0）。 | 温度の効果がないときの基準。 |
| H40 | H で E を文献値 40 kJ/mol に固定。 | 文献値が、このデータに合うかの確認。 |
| F | H と同じ形で、T に流動先端温度 Tff を使う。 | 参考。Tff は流れの結果で、成形条件から事前には分からない。 |
| C | 構造なし。Tset を、ほかの入力と並べた通常の GPR（R1）。 | 構造が要るかの基準。構造あり（A・B）が C と同等なら、構造が支持された、とみなす。 |

図0に、これらの形の違いの模式図（実データではない）を付けた。

## 使い方（このノートを一人で実行して、結果を判断するために）

1. メニューの「すべて実行」で、上から順に実行する（約 40 分）。カーネルは、このプロジェクトの `.venv` を選ぶ。
2. 設定を変えたいときは、最初の「設定エリア」のセルだけを直して、最初から実行し直す（高解像度のグラフの設定は、下の 7）。
3. 各節の最後に、**判定**が出る（`>> 判定：OK` か `>> 判定：★要確認`）。OK は「想定どおり」、★ は「想定と違う、または結果を読むときに注意が要る点」。基準は、判定の文の中に書いてある。★ が出ても、エラーではない。説明書の「判定の読み方」で、★ の意味を確かめる。
4. 図と表は `results/step5a/` に保存される。図の見方は、説明書 `results/学習結果説明_手順5a.pdf` にある。
5. データを取り直したとき、問題のあるデータを入れ替えたときは、`問題データ一覧.md` の手順に従う。このノートは、全条件のファイルを読み、`02_機械学習データ/問題データ一覧.csv` の「除外」を除く。
6. 用語や図の見方で分からないところは、説明書の用語集で調べる。記号（Tff、Tset、R3、E_h など）の意味は、`results/学習結果説明_記号表.pdf` に一覧がある。
7. **高解像度のグラフ**（報告書・発表資料用。GPR6月ノートのセル9-2 と同じ見せ方）は、「高解像度のグラフ」の節のセルで作る。**このセルだけは、セルの先頭にも設定エリアがあり、そこだけを直して、このセルだけを実行し直せる**（学習はやり直さない。全体を一度実行した後で）。保存先は `results/step5a/highres/`（Git には入れない）。

## 1. 設定

In [ ]:
# セル1：設定エリア  ▼▼▼ 変えてよいのは、ここだけ ▼▼▼
SEED = 42                     # 乱数シード
NOISE_LB = 1e-6               # ノイズ分散の下限（旧ノートは 1e-3）
N_RESTARTS = 5                # GPR の最適化のやり直し回数（増やすと遅いが、安定する）
USE_EXCLUSIONS = True         # True：問題データ一覧の「除外」を使う／False：全条件を使う（取り直した後の確認用）
REPO = ".."                   # リポジトリ直下（このノートは 01_機械学習コードフォルダ にある）
FIG_DPI = 200                 # 保存する PNG の解像度
import numpy as np             # 格子の指定に使う
MAIN_STRUCT = "B"             # 本命の構造（ユーザー決定、2026-10-12）。H・A・Hz・B・N・H40・F・C のどれでも
ALT_STRUCT = "A"              # 対案の構造（同上）。「8. 本命と対案の比較」と、高解像度のグラフの既定で使う
MAIN_DATA = "なし"            # 主に使う外れ値の扱い
TEMP_HOLDOUT = 210            # S_T でテストにする設定温度 [℃]
E_GRID = np.arange(-20.0, 62.5, 2.5)        # 1 次元の E の格子 [kJ/mol]（H・A・Hz・F）
EH_GRID = np.arange(-40.0, 45.0, 5.0)       # B の横方向の E_h の格子 [kJ/mol]
EV_GRID = np.arange(-20.0, 45.0, 5.0)       # B の縦方向の E_v の格子 [kJ/mol]
N_RESTARTS_PROFILE = 3        # 尤度の格子の各点での最適化のやり直し回数
N_RESTARTS_FINAL = 3          # 予測の評価のための学習でのやり直し回数
N_JOBS = 6                    # 並列に計算する数（CPU の数に合わせる）
E_LIT = 40.0                  # 文献値の目安 [kJ/mol]（PP の流動の活性化エネルギー）
# ▲▲▲ 設定エリア終了 ▲▲▲

In [ ]:
# セル1.5：ライブラリ、フォント、出力先
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from joblib import Parallel, delayed

OUT_DIR = os.path.join(REPO, "results", "step5a")
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 80
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
try:
    fm.findfont("IPAexGothic", fallback_to_default=False)
except ValueError:
    raise RuntimeError("日本語フォント IPAexGothic が見つかりません。インストールしてからカーネルを再起動してください。")
plt.rcParams["font.family"] = ["IPAexGothic", "DejaVu Sans"]
print("Python:", sys.executable, "/", sys.version.split()[0])
print("出力先:", os.path.abspath(OUT_DIR))

### 共通の部品

データの読み込み、問題データ一覧の「除外」、評価、データの分け方の関数。中身は `04_補助スクリプト/nb_common.py` と同じ。

In [ ]:
# セル：共通の部品（読み込み・モデル・評価）。04_補助スクリプト/nb_common.py の中身。直すときは、そのファイルを直して、ノートを作り直す
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, DotProduct, Matern, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------- 設定（ノートの設定エリアの値があれば、それを使う）
SEED = globals().get("SEED", 42)
NOISE_LB = globals().get("NOISE_LB", 1e-6)                 # ノイズ分散の下限（CLAUDE.md §5。旧ノートは 1e-3）
N_RESTARTS = globals().get("N_RESTARTS", 5)                # GPR の最適化のやり直し回数
REPO = globals().get("REPO", os.environ.get("NB_REPO", ".."))   # リポジトリ直下（ノートは 01_機械学習コードフォルダ にあるので ".."。スクリプトは環境変数 NB_REPO で指定）
USE_EXCLUSIONS = globals().get("USE_EXCLUSIONS", True)     # 問題データ一覧の「除外」を使うか
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # 下限への張り付きは kernels.csv に記録する

# ---------------------------------------------------------------- データの場所
DATA_DIR = os.path.join(REPO, "02_機械学習データ", "NR500")
TEST_DIR = os.path.join(REPO, "03_予測検証用テストデータ", "NR500")
PROBLEM_CSV = os.path.join(REPO, "02_機械学習データ", "問題データ一覧.csv")
MATS = {"PP": "PP", "MIX": "粉砕材混合PP"}
FILES = {   # (材料, 組): (上位フォルダ, 中間フォルダ, フォルダ, ファイル名)。どれも全条件のファイル（問題のある条件は、一覧で除く）
    ("PP", "5data"): (DATA_DIR, "02_バージンPP再々実験_5data", "03_pp_max_5data", "結合_03_pp_max_5data_外れ値{o}.csv"),
    ("PP", "3data"): (TEST_DIR, "02_バージンPP再々実験_5data用検証データ(3data)", "02_pp_max_3data", "結合_02_pp_max_3data_外れ値{o}.csv"),
    ("PP", "Alldata"): (DATA_DIR, "01_バージンPP再々実験_alldata", "01_pp_max_Alldata", "結合_01_pp_max_Alldata_外れ値{o}.csv"),
    ("MIX", "5data"): (DATA_DIR, "04_粉砕材混合PP実験_5data", "03_mixpp_max_5data", "結合_03_mixpp_max_5data_外れ値{o}.csv"),
    ("MIX", "3data"): (TEST_DIR, "04_粉砕材混合PP実験_5data用検証データ(3data)", "02_mixpp_max_3data", "結合_02_mixpp_max_3data_外れ値{o}.csv"),
    ("MIX", "Alldata"): (DATA_DIR, "03_粉砕材混合PP実験_alldata", "01_mixpp_max_Alldata", "結合_01_mixpp_max_Alldata_外れ値{o}.csv"),
}
LEGACY_FILES = {   # 1.5 mm・3 mm/s を除いてあるファイル（no1.5-3.0）。一覧の「除外」で同じデータになるかの確認用
    ("PP", "3data"): (TEST_DIR, "02_バージンPP再々実験_5data用検証データ(3data)", "05_pp_max_3data(no1.5-3.0)", "結合_05_pp_max_3data(no1.5-3.0)_外れ値{o}.csv"),
    ("PP", "Alldata"): (DATA_DIR, "01_バージンPP再々実験_alldata", "04_pp_max_Alldata(no1.5-3.0)", "結合_04_pp_max_Alldata(no1.5-3.0)_外れ値{o}.csv"),
}
COLS = {"元ファイル": "file", "キャビ厚": "H", "射出速度[mm/s]": "v", "樹脂粘度η34(Pa・s)": "eta", "せん断速度γ34(1/s)": "gd",
        "流動中樹脂圧力P3_1(MPa)": "P31", "流動中樹脂圧力P3_2(MPa)": "P32", "加熱シリンダ温度[℃]": "Tset", "流動中樹脂温度(℃)": "Tff"}
RE_SRC = r"^(\d{3})_(\d+)℃_(\d{3})_([\d.]+)mm\.csv$"
D = {}                  # (材料, 組, 外れ値) → DataFrame。ノートで D = load_all() として入れる
REMOVED = {}            # (材料, 組, 外れ値) → 一覧の「除外」で除いた行数


def path_of(mat, grp, o, files=None):
    top, d1, d2, fn = (files or FILES)[(mat, grp)]
    return os.path.join(top, d1, d2, f"外れ値{o}", fn.format(o=o))


# ---------------------------------------------------------------- 問題データ一覧
def read_problem_list(path=None):
    """問題データ一覧.csv を読む（空欄は空文字）"""
    return pd.read_csv(path or PROBLEM_CSV, encoding="utf-8-sig", dtype=str).fillna("")


def problem_mask(df, mat, grp, plist, kinds):
    """区分が kinds のどれかで、再測定が「済み」でない一覧の行のうち、この材料・組・条件に当てはまる行の真偽値と、その ID を返す"""
    mask = np.zeros(len(df), bool)
    ids = [[] for _ in range(len(df))]
    for _, r in plist.iterrows():
        if r["区分"] not in kinds or r["再測定"] == "済み" or r["データ"] not in ("新", ""):
            continue
        if r["材料"] not in ("すべて", "", MATS[mat]) or r["組"] not in ("", grp):
            continue
        m = np.ones(len(df), bool)
        if r["設定温度"] != "":
            m &= df["Tset"].to_numpy() == float(r["設定温度"])
        if r["厚み"] != "":
            m &= np.isclose(df["H"].to_numpy(), float(r["厚み"]))
        if r["射出速度"] != "":
            m &= df["v"].to_numpy() == float(r["射出速度"])
        if r.get("ショット", "") != "":                      # 空欄は条件全体。数字なら、そのショット（元ファイル名の先頭 3 桁）だけ
            m &= (df["shot"].to_numpy() == int(r["ショット"])) if "shot" in df.columns else np.zeros(len(df), bool)
        mask |= m
        for i in np.flatnonzero(m):
            ids[i].append(r["ID"])
    return mask, [";".join(x) for x in ids]


def load(mat, grp, o, plist=None, exclude=None):
    """結合ファイルを読み、形式を確かめ、問題データ一覧の「除外」を除く。列 sens には、「感度確認」の ID が入る（なければ空文字）"""
    exclude = USE_EXCLUSIONS if exclude is None else exclude
    df = pd.read_csv(path_of(mat, grp, o), encoding="utf-8-sig")
    assert list(df.columns) == list(COLS), f"列名が想定と違う: {list(df.columns)}"
    df = df.rename(columns=COLS)
    assert df.isna().sum().sum() == 0, "欠損がある"
    p = df["file"].str.extract(RE_SRC)
    assert p.notna().all().all(), "ファイル名の形式に合わない行がある"
    assert np.allclose(p[1].astype(float), df["Tset"]) and np.allclose(p[2].astype(float), df["v"]) \
        and np.allclose(p[3].astype(float), df["H"]), "ファイル名と列の値が一致しない"
    df["shot"] = p[0].astype(int)
    df["Tset"] = df["Tset"].astype(float)
    plist = read_problem_list() if plist is None else plist
    n0 = len(df)
    if exclude:
        drop, _ = problem_mask(df, mat, grp, plist, ("除外",))
        df = df[~drop].reset_index(drop=True)
    REMOVED[(mat, grp, o)] = n0 - len(df)
    _, df["sens"] = problem_mask(df, mat, grp, plist, ("感度確認",))
    df["is_sens"] = df["sens"] != ""
    df["cond"] = df["Tset"].astype(int).astype(str) + "_" + df["H"].astype(str) + "_" + df["v"].astype(int).astype(str)
    df["ln_eta"], df["ln_gd"], df["ln_H"] = np.log(df["eta"]), np.log(df["gd"]), np.log(df["H"])
    df["ln_P31"], df["ln_P32"] = np.log(df["P31"] * 1e6), np.log(df["P32"] * 1e6)   # CSV の圧力は MPa。Pa に直してから ln をとる
    df["ln_v"] = np.log(df["v"])
    df["ln_Tff"] = np.log(df["Tff"] + 273.15)          # 流動先端温度 [K]
    for c in ["gd", "Tff", "P32"]:   # 条件ごとの平均（成形条件から見たばらつきを見るモデル用。cm は condition mean）
        df[f"ln_{c}_cm"] = df.groupby("cond")[f"ln_{c}"].transform("mean")
    return df


def load_all(exclude=None):
    """全材料・全組・外れ値なし/あり を読む。学習と検証で条件が重ならないことも確かめる"""
    plist = read_problem_list()
    out = {(mat, grp, o): load(mat, grp, o, plist, exclude) for mat in MATS for grp in ["5data", "3data", "Alldata"] for o in ["なし", "あり"]}
    for mat in MATS:
        for o in ["なし", "あり"]:
            assert not (set(out[(mat, "5data", o)]["cond"]) & set(out[(mat, "3data", o)]["cond"])), "学習と検証で条件が重なっている"
    return out


def legacy_check(D_):
    """全条件のファイルから一覧で除いたデータが、1.5 mm・3 mm/s を除いてあるファイル（no1.5-3.0）と同じ行か（PP の 3data・Alldata）"""
    rows = []
    for (mat, grp), _ in LEGACY_FILES.items():
        for o in ["なし", "あり"]:
            old = pd.read_csv(path_of(mat, grp, o, LEGACY_FILES), encoding="utf-8-sig")
            same = set(old["元ファイル"]) == set(D_[(mat, grp, o)]["file"])
            rows.append({"材料": MATS[mat], "組": grp, "外れ値": o, "一覧で除いた後の行数": len(D_[(mat, grp, o)]), "no1.5-3.0 の行数": len(old), "同じ行": same})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- モデル
def kernel_for(d, linear=False):
    """ARD Matern 5/2 ＋ ノイズ。linear=True なら、線形カーネル（DotProduct）も足す（低速側への外挿のため。CLAUDE.md §5）"""
    k = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0] * d, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    if linear:
        k = k + C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e3))
    return k + WhiteKernel(noise_level=0.1, noise_level_bounds=(NOISE_LB, 1e1))


def make_gpr(d, linear=False):
    return GaussianProcessRegressor(kernel=kernel_for(d, linear), n_restarts_optimizer=N_RESTARTS, random_state=SEED)


def fit_ols(X_tr, y_tr):
    """切片ありの最小二乗回帰。係数、(XᵀX)⁻¹、残差分散を返す"""
    A_tr = np.column_stack([np.ones(len(X_tr)), X_tr])
    beta, *_ = np.linalg.lstsq(A_tr, y_tr, rcond=None)
    resid = y_tr - A_tr @ beta
    return beta, np.linalg.inv(A_tr.T @ A_tr), resid @ resid / (len(y_tr) - A_tr.shape[1])


def predict_ols(model, X):
    """予測平均と、新しい1ショットに対する予測標準偏差"""
    beta, XtX_inv, s2 = model
    A_new = np.column_stack([np.ones(len(X)), X])
    lev = np.einsum("ij,jk,ik->i", A_new, XtX_inv, A_new)
    return A_new @ beta, np.sqrt(s2 * (1.0 + lev))


MODELS = {}             # 記号 → {"kind": "ols" か "gpr", "cols": [入力の列], "linear": 線形カーネルを足すか}。ノートで決める


def fit_model(key, tr):
    spec = MODELS[key]
    X, y = tr[spec["cols"]].to_numpy(), tr["ln_eta"].to_numpy()
    if spec["kind"] == "ols":
        return {"kind": "ols", "model": fit_ols(X, y)}
    sc = StandardScaler().fit(X)              # 標準化と中心化は学習データだけで計算する
    y0 = y.mean()
    gp = make_gpr(X.shape[1], spec.get("linear", False)).fit(sc.transform(X), y - y0)
    return {"kind": "gpr", "gp": gp, "sc": sc, "y0": y0}


def predict_model(key, fit, te):
    """予測平均、予測標準偏差（ノイズ込み）、潜在関数の標準偏差、ノイズの標準偏差"""
    X = te[MODELS[key]["cols"]].to_numpy()
    if fit["kind"] == "ols":
        mu, sd = predict_ols(fit["model"], X)
        return mu, sd, np.full(len(X), np.nan), float(np.sqrt(fit["model"][2]))
    mu, sd = fit["gp"].predict(fit["sc"].transform(X), return_std=True)
    noise = fit["gp"].kernel_.k2.noise_level
    return mu + fit["y0"], sd, np.sqrt(np.maximum(sd ** 2 - noise, 0.0)), float(np.sqrt(noise))


def evaluate(y, mu, sd):
    """ln η の指標：R2、RMSE、95% 予測区間の被覆率、NLPD（各ショットの予測分布の負の対数密度の平均）、予測標準偏差の平均"""
    z = (y - mu) / sd
    return {"R2": r2_score(y, mu) if len(y) > 1 else np.nan, "RMSE": float(np.sqrt(mean_squared_error(y, mu))),
            "coverage95": float(np.mean(np.abs(z) <= 1.96)),
            "NLPD": float(np.mean(0.5 * np.log(2 * np.pi * sd ** 2) + 0.5 * z ** 2)), "mean_sd": float(np.mean(sd))}


def kernel_row(key, fit):
    """学習した GPR のハイパーパラメータ（振幅、長さスケール、ノイズ、線形項、対数周辺尤度、下限・上限への張り付き）"""
    gp = fit["gp"]
    k = gp.kernel_
    lin = MODELS[key].get("linear", False)
    prod, lin_prod = (k.k1.k1, k.k1.k2) if lin else (k.k1, None)
    ls = np.atleast_1d(prod.k2.length_scale)
    row = {"amplitude": float(np.sqrt(prod.k1.constant_value)), "noise_var": float(k.k2.noise_level),
           "noise_sd": float(np.sqrt(k.k2.noise_level)), "LML": float(gp.log_marginal_likelihood_value_),
           "noise_at_lower_bound": bool(k.k2.noise_level < NOISE_LB * 1.05),
           "ls_at_bound": bool(np.any(ls > 99) or np.any(ls < 0.0105))}
    if lin_prod is not None:
        row.update({"lin_amplitude": float(np.sqrt(lin_prod.k1.constant_value)), "lin_sigma0": float(lin_prod.k2.sigma_0)})
    row.update({f"ls_{c}": float(l) for c, l in zip(MODELS[key]["cols"], ls)})
    return row


def within_stats(df):
    """同じ条件内のばらつき（プール）。dof：自由度、var_after_gd：ln γ̇ で説明した残りの分散"""
    g = df.groupby("cond")
    dof = int((g.size() - 1).sum())
    de = df["ln_eta"] - g["ln_eta"].transform("mean")
    dg = df["ln_gd"] - g["ln_gd"].transform("mean")
    var_e = float((de ** 2).sum() / dof)
    b = float((de * dg).sum() / (dg ** 2).sum())
    return {"dof": dof, "var_ln_eta": var_e, "sd_ln_eta": var_e ** 0.5, "corr_with_ln_gd": float((de * dg).sum() / np.sqrt((de ** 2).sum() * (dg ** 2).sum())),
            "slope_vs_ln_gd": b, "var_after_gd": float(((de - b * dg) ** 2).sum() / (dof - 1))}


# ---------------------------------------------------------------- データの分け方
def split_V(mat, o):
    """5data で学習し、3data で検証する（射出速度の両端の外挿）。te の group 列は、低速側（3・5 mm/s）か高速側（280・320 mm/s）"""
    tr, te = D[(mat, "5data", o)].copy(), D[(mat, "3data", o)].copy()
    te["group"] = np.where(te["v"] <= 5, "低速側", np.where(te["v"] >= 280, "高速側", "その他"))
    assert (te["group"] != "その他").all(), "検証データに、低速側・高速側のどちらでもない条件がある"
    return tr, te


def split_random(df):
    """行単位のランダム 20%（従来の分け方。random_state = SEED）"""
    idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED)
    te = df.iloc[idx_te].copy()
    te["group"] = "全体"
    return df.iloc[idx_tr].copy(), te


def split_T(mat, o, t_test=210):
    """設定温度 t_test の全条件をテストにし、5data の残りの温度で学習する（温度の内挿の検証）。3data の同じ温度は te3 として別に返す"""
    d5, d3 = D[(mat, "5data", o)], D[(mat, "3data", o)]
    tr, te = d5[d5["Tset"] != t_test].copy(), d5[d5["Tset"] == t_test].copy()
    te["group"] = "全体"
    te3 = d3[d3["Tset"] == t_test].copy()
    te3["group"] = np.where(te3["v"] <= 5, "低速側", "高速側")
    return tr, te, te3


def pool_of(mat, o):
    """5data と 3data をまとめたデータ"""
    return pd.concat([D[(mat, "5data", o)], D[(mat, "3data", o)]], ignore_index=True)


# ---------------------------------------------------------------- 学習と評価の記録
METRICS, KERNELS, OLSROWS, PRED, FITS = [], [], [], {}, {}      # FITS：(材料, split, データ, モデル) → 学習した予測器
SENS_SUFFIX = "（感度確認の条件を除く）"


def _record(mat, name, data, key, fit, tr, te, pure):
    """学習した予測器 fit で te を予測し、指標（subset ごと）と予測を記録する"""
    subsets = {"全体": np.ones(len(te), bool)}
    for g in te["group"].unique():
        if g != "全体":
            subsets[g] = (te["group"] == g).to_numpy()
    if te["is_sens"].any():
        subsets.update({s + SENS_SUFFIX: m & ~te["is_sens"].to_numpy() for s, m in list(subsets.items())})
    y_te = te["ln_eta"].to_numpy()
    mu, sd, sd_f, noise_sd = predict_model(key, fit, te)
    PRED[(mat, name, data, key)] = pd.DataFrame({"y": y_te, "mu": mu, "sd": sd, "sd_f": sd_f, "v": te["v"].to_numpy(), "H": te["H"].to_numpy(),
                                                 "Tset": te["Tset"].to_numpy(), "cond": te["cond"].to_numpy(), "group": te["group"].to_numpy(),
                                                 "is_sens": te["is_sens"].to_numpy()})
    for sub, mask in subsets.items():
        if mask.sum() == 0:
            continue
        ev = evaluate(y_te[mask], mu[mask], sd[mask])
        METRICS.append({"material": MATS[mat], "split": name, "data": data, "model": key, "subset": sub, "n_train": len(tr),
                        "n_test": int(mask.sum()), "n_cond_test": int(te.loc[mask, "cond"].nunique()), **ev,
                        "mean_sd_latent": float(np.nanmean(sd_f[mask])) if fit["kind"] == "gpr" else np.nan, "noise_sd": noise_sd})


def run_split(mat, split, data, tr, te, keys=None, tag="", extra_te=None):
    """全モデルを学習し、指標（subset ごと）・カーネル・予測を記録する。te には group 列がある。
    te に「感度確認」の条件があれば、各 subset から除いた指標（subset 名の後ろに SENS_SUFFIX）も記録する。
    tag は split 名の後ろに付ける印。extra_te（{split 名: テスト}）があれば、同じ学習で、それらも評価する"""
    keys = keys or ALL_KEYS
    name = split + tag
    tests = {name: te, **{s + tag: t for s, t in (extra_te or {}).items()}}
    pure = within_stats(tr)
    for key in keys:
        t0 = time.time()
        fit = fit_model(key, tr)
        FITS[(mat, name, data, key)] = fit
        for nm, te_ in tests.items():
            _record(mat, nm, data, key, fit, tr, te_, pure)
        base = {"material": MATS[mat], "split": name, "data": data, "model": key, "n_train": len(tr)}
        if fit["kind"] == "gpr":
            KERNELS.append({**base, **kernel_row(key, fit), "pure_sd_ln_eta": pure["sd_ln_eta"], "pure_sd_after_gd": pure["var_after_gd"] ** 0.5})
        else:
            names = ["intercept"] + MODELS[key]["cols"]
            OLSROWS.append({**base, "resid_sd": float(np.sqrt(fit["model"][2])), **dict(zip(names, map(float, fit["model"][0])))})
        print(f"{MATS[mat]} {name} {data} {key}: 学習 {len(tr)} 行、{time.time() - t0:.1f} 秒")


# ---------------------------------------------------------------- 条件を1つずつ抜く交差検証（LOCO）
def loco_gpr(key, pool):
    """GPR：ハイパーパラメータを全データで決めて固定し、条件を抜いた予測を閉じた式で計算する（平均 = y_g − [K⁻¹_gg]⁻¹(K⁻¹y)_g、共分散 = [K⁻¹_gg]⁻¹）"""
    spec = MODELS[key]
    X, y = pool[spec["cols"]].to_numpy(), pool["ln_eta"].to_numpy()
    sc = StandardScaler().fit(X)
    Xs, yc = sc.transform(X), y - y.mean()
    gp = make_gpr(X.shape[1], spec.get("linear", False)).fit(Xs, yc)
    Kinv = np.linalg.inv(gp.kernel_(Xs))              # ノイズ込みの共分散行列の逆行列
    alpha = Kinv @ yc
    mu, sd = np.empty(len(y)), np.empty(len(y))
    for idx in pool.groupby("cond").indices.values():
        Minv = np.linalg.inv(Kinv[np.ix_(idx, idx)])
        mu[idx] = y[idx] - Minv @ alpha[idx]
        sd[idx] = np.sqrt(np.diag(Minv))
    return mu, sd, gp


def loco_ols(key, pool):
    """線形回帰：条件ごとに、その条件を除いて当てはめ直す"""
    cols = MODELS[key]["cols"]
    X, y = pool[cols].to_numpy(), pool["ln_eta"].to_numpy()
    mu, sd = np.empty(len(y)), np.empty(len(y))
    for idx in pool.groupby("cond").indices.values():
        keep = np.ones(len(y), bool)
        keep[idx] = False
        mu[idx], sd[idx] = predict_ols(fit_ols(X[keep], y[keep]), X[idx])
    return mu, sd


# ---------------------------------------------------------------- 判定の表示（GPR6月ノートの「判定」にならう）
def judge(ok, text):
    """結果の判定を1行で表示する。ok が True なら「OK」、False なら「要確認」"""
    print(f">> 判定：{'OK' if ok else '★要確認'}　{text}")


# ---------------------------------------------------------------- 高解像度のグラフ（GPR6月ノートのセル9-2 にならう。報告書・発表資料用）
HR_COLORS = {"H": {0.5: "#d62728", 1.0: "#1f77b4", 1.5: "#f0b400", 2.0: "#2ca02c"},       # 厚みごとの色（6月のノートと同じ並び。黄は、線で見やすい濃さにした）
             "Tset": {190: "#1f77b4", 210: "#2ca02c", 230: "#d62728"}}                       # 設定温度ごとの色
HR_MARKS = {"H": {0.5: "o", 1.0: "s", 1.5: "^", 2.0: "D"}, "Tset": {190: "o", 210: "s", 230: "^"}}
HR_DEFAULT = dict(layouts=["thickness", "temperature"], band="predict", points="cond", show_test=True, mark_sens=True, x_range=None, extend=1.3,
                  xlim={"thickness": None, "temperature": (30, 50000)}, ylim={"thickness": None, "temperature": (0.5, 5000)}, figsize=(15, 10), dpi=600, font={"axis": 24, "legend": 24, "title": 24, "tick": 28},
                  marker=200, marker_shot=70, line=3.0, show_title=True, show_xlabel=False, show_ylabel=False, formats=["png"], show=False, n_grid=120)


def hr_split_conds(mat, data, split, t_test=210):
    """学習に使った条件の集合。S_V：5data／S_T：5data の t_test 以外の温度／ALL：5data と 3data"""
    d5, d3 = D[(mat, "5data", data)], D[(mat, "3data", data)]
    if split == "S_V":
        return set(d5["cond"])
    if split == "S_T":
        return set(d5.loc[d5["Tset"] != t_test, "cond"])
    return set(d5["cond"]) | set(d3["cond"])


def hr_inputs(sub, lg, h, t):
    """予測線の入力（ln γ̇ = lg の点）。Tff・P3_2・射出速度は、同じ設定温度・厚みの条件平均を ln γ̇ で内挿する（どの入力のモデルでも描けるように）"""
    cm = sub.groupby("cond").agg(lg=("ln_gd", "mean"), tff=("ln_Tff", "mean"), p32=("ln_P32", "mean"), lv=("ln_v", "mean")).sort_values("lg")
    ip = lambda c: np.interp(lg, cm["lg"], cm[c])
    return pd.DataFrame({"ln_gd": lg, "ln_gd_cm": lg, "ln_H": np.log(h), "Tset": float(t), "ln_Tff": ip("tff"), "ln_Tff_cm": ip("tff"),
                         "ln_P32": ip("p32"), "ln_P32_cm": ip("p32"), "ln_v": ip("lv")})


def hr_flow(predict, mat, data, train_conds, title_model, stem, out_dir, **kw):
    """せん断速度 ― 粘度の学習結果グラフを、高解像度で 1 枚ずつ保存する（6月のノートのセル9-2 と同じ見せ方）。
    predict(Xg) → (予測平均 ln η, 予測標準偏差（ノイズ込み）, 潜在関数の標準偏差)。Xg は hr_inputs の DataFrame。
    train_conds：学習に使った条件の集合（それ以外は、検証の点として白抜きで重ねる）。設定は HR_DEFAULT を参照（kw で上書き）。保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    cfg = {**HR_DEFAULT, **kw}
    os.makedirs(out_dir, exist_ok=True)
    pool = pool_of(mat, data)
    curves = {}                                                         # (厚み, 設定温度) → (x, 予測平均, 帯の幅の標準偏差)。レイアウトが違っても同じ線なので、1 回だけ計算する
    for (h, t), sub in pool.groupby(["H", "Tset"]):
        lgc = sub.groupby("cond")["ln_gd"].mean()
        if len(lgc) < 2:
            continue
        lo, hi = (np.log(cfg["x_range"][0]), np.log(cfg["x_range"][1])) if cfg["x_range"] else (lgc.min() - np.log(cfg["extend"]), lgc.max() + np.log(cfg["extend"]))
        lg = np.linspace(lo, hi, cfg["n_grid"])
        mu, sd, sdf = predict(hr_inputs(sub, lg, h, t))
        curves[(float(h), int(t))] = (np.exp(lg), np.asarray(mu), np.asarray(sd if cfg["band"] == "predict" else sdf))
    saved = []
    ft = cfg["font"]
    band_name = "95% 予測区間" if cfg["band"] == "predict" else "95% 信頼区間"
    for layout in cfg["layouts"]:
        by, sk = ("H", "Tset") if layout == "thickness" else ("Tset", "H")        # by：1 枚ごとに固定する量、sk：1 枚の中で線を分ける量
        for gv in sorted({k[0] if by == "H" else k[1] for k in curves}):
            fig, ax = plt.subplots(figsize=cfg["figsize"], dpi=cfg["dpi"])
            series, any_test, any_sens, xs, ys = [], False, False, [], []
            for sv in sorted({(k[1] if by == "H" else k[0]) for k in curves if (k[0] if by == "H" else k[1]) == gv}):
                h, t = (gv, sv) if by == "H" else (sv, gv)
                col, mk = HR_COLORS[sk][sv], HR_MARKS[sk][sv]
                x, mu, s = curves[(float(h), int(t))]
                if cfg["band"] in ("predict", "latent"):
                    ax.fill_between(x, np.exp(mu - 1.96 * s), np.exp(mu + 1.96 * s), color=col, alpha=0.22, zorder=15)
                    ax.plot(x, np.exp(mu - 1.96 * s), color=col, lw=0.6, alpha=0.7, zorder=16)
                    ax.plot(x, np.exp(mu + 1.96 * s), color=col, lw=0.6, alpha=0.7, zorder=16)
                ax.plot(x, np.exp(mu), color=col, lw=cfg["line"], zorder=20)
                sub = pool[(np.isclose(pool["H"], h)) & (pool["Tset"] == t)]
                if cfg["points"] == "cond":
                    a = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean"), sens=("is_sens", "max")).reset_index()
                    px, py, tr_m, se_m, msize = np.exp(a["lg"].to_numpy()), np.exp(a["le"].to_numpy()), a["cond"].isin(train_conds).to_numpy(), a["sens"].to_numpy().astype(bool), cfg["marker"]
                else:
                    px, py, tr_m, se_m, msize = sub["gd"].to_numpy(), sub["eta"].to_numpy(), sub["cond"].isin(train_conds).to_numpy(), sub["is_sens"].to_numpy().astype(bool), cfg["marker_shot"]
                xs += list(x) + list(px)
                ys += list(np.exp(mu)) + list(py)
                ax.scatter(px[tr_m], py[tr_m], s=msize, color=col, marker=mk, alpha=0.85, edgecolors="k", linewidths=0.6, zorder=14 - list(HR_COLORS[sk]).index(sv))
                if cfg["show_test"] and (~tr_m).any():
                    ax.scatter(px[~tr_m], py[~tr_m], s=msize, facecolors="white", edgecolors=col, marker=mk, linewidths=2.5, zorder=14 - list(HR_COLORS[sk]).index(sv))
                    any_test = True
                if cfg["mark_sens"] and se_m.any():
                    ax.scatter(px[se_m], py[se_m], s=msize * 1.3, color="k", marker="x", linewidths=3, zorder=25)
                    any_sens = True
                series.append((sv, col, mk))
            head = "キャビティ厚み" if sk == "H" else "設定温度"
            handles = [Patch(facecolor="gray", alpha=0.3, label=band_name), Patch(color="none", label=head)]
            handles += [Line2D([], [], color=col, lw=cfg["line"], marker=mk, markersize=np.sqrt(cfg["marker"]), markeredgecolor="k", markeredgewidth=0.6,
                               label=(f"{sv} mm" if sk == "H" else f"{int(sv)} ℃")) for sv, col, mk in series]
            if any_test:
                handles.append(Line2D([], [], color="gray", ls="none", marker="o", markersize=np.sqrt(cfg["marker"]), markerfacecolor="white", markeredgewidth=2.5, label="検証の条件（白抜き）"))
            if any_sens:
                handles.append(Line2D([], [], color="k", ls="none", marker="x", markersize=np.sqrt(cfg["marker"]), markeredgewidth=3, label="感度確認の条件"))
            if cfg["band"] not in ("predict", "latent"):
                handles = handles[1:]
            ax.legend(handles=handles, loc="upper right", fontsize=ft["legend"], framealpha=1.0, edgecolor="black")
            ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=ft["tick"], pad=8)
            ax.set_xscale("log")
            ax.set_yscale("log")
            xl = cfg["xlim"][layout] if isinstance(cfg["xlim"], dict) else cfg["xlim"]       # 範囲：(下限, 上限)＝固定／None＝その図のデータに合わせる
            yl = cfg["ylim"][layout] if isinstance(cfg["ylim"], dict) else cfg["ylim"]
            auto_x, auto_y = xl is None, yl is None
            ax.set_xlim(*((min(xs) / 1.4, max(xs) * 1.4) if auto_x else xl))
            ax.set_ylim(*((min(ys) / 1.5, max(ys) * 1.5) if auto_y else yl))
            for axis, auto in ((ax.xaxis, auto_x), (ax.yaxis, auto_y)):
                if auto:                               # 範囲が狭いので、1・2・5 × 10^n に目盛りを付けて、普通の数で表す
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))
                    axis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
                else:                                  # 6月のノートと同じ（10 の累乗の目盛り）
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
                    axis.set_major_formatter(ticker.LogFormatterSciNotation())
                axis.set_minor_formatter(ticker.NullFormatter())
            ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
            ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
            gtxt = f"厚み {gv} mm" if by == "H" else f"設定温度 {int(gv)} ℃"
            if cfg["show_title"]:
                ax.set_title(f"{MATS[mat]}　せん断速度と粘度の関係（{gtxt}）\n{title_model}", fontsize=ft["title"], pad=15)
            if cfg["show_xlabel"]:
                ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ (1/s)", fontsize=ft["axis"], labelpad=15)
            if cfg["show_ylabel"]:
                ax.set_ylabel("樹脂粘度 (Pa·s)", fontsize=ft["axis"], labelpad=15)
            fig.tight_layout()
            base = f"{stem}_{'by-thickness' if by == 'H' else 'by-temperature'}_{(str(gv) + 'mm') if by == 'H' else (str(int(gv)) + 'C')}"
            for fmt in cfg["formats"]:
                path = os.path.join(out_dir, f"{base}.{fmt}")
                fig.savefig(path, bbox_inches="tight", dpi=cfg["dpi"])
                saved.append(path)
            if cfg["show"]:
                plt.show()
            plt.close(fig)
    return saved

## 2. データの読み込み

In [ ]:
# セル2：読み込み
D = load_all()
leg = legacy_check(D)
judge(bool(leg["同じ行"].all()) or not USE_EXCLUSIONS, "一覧の「除外」を除いたデータが、これまでの no1.5-3.0 のファイルと同じ（取り直した後は違ってよい）")
for mat in MATS:
    tr, te = D[(mat, "5data", MAIN_DATA)], D[(mat, "3data", MAIN_DATA)]
    print(f"{MATS[mat]}：学習（5data）{len(tr)} 行 {tr['cond'].nunique()} 条件、検証（3data）{len(te)} 行 {te['cond'].nunique()} 条件、感度確認の条件（学習 {int(tr['is_sens'].sum())} 行、検証 {int(te['is_sens'].sum())} 行）")

## 3. シフト構造の定義と学習の関数

- `design`：データ → (標準化の定数を決めるための入力, 実際の入力, 出力のシフト量)。
  - 構造あり（shift）：入力は (u, w) = (ln γ̇ ＋ ln a(T; E_h), −(ln γ̇ ＋ 3 ln H))。出力のシフト量は ln a(T; E_v)。T は Tset [K]（基準が Tff のときは Tff [K]）。
  - 構造なし（free）：入力は (ln γ̇, ln H, Tset)。シフトなし。
- `fit_variant`・`predict_variant`：学習と予測。標準化・中心化は学習データだけで計算し、E に依存しない定数を使う。

In [ ]:
# セル3：構造の定義、学習・予測の関数
R_GAS, TREF_K = 8.314, 483.15                     # 気体定数 [J/mol/K]、基準温度 210 ℃ [K]
MODE_COLS = {"cm": ("ln_gd_cm", "ln_Tff_cm"), "shot": ("ln_gd", "ln_Tff")}    # cm：条件平均の γ̇、shot：ショットごとの γ̇


def shift_ln(Tk, E):
    """ln a(T; E) = (E/R)(1/T − 1/Tref)。E は kJ/mol、Tk は K"""
    return (E * 1000.0 / R_GAS) * (1.0 / np.asarray(Tk) - 1.0 / TREF_K)


def design(df, v, mode):
    gcol, fcol = MODE_COLS[mode]
    lng = df[gcol].to_numpy()
    if v["kind"] == "free":
        X = np.column_stack([lng, df["ln_H"].to_numpy(), df["Tset"].to_numpy()])
        return X, X, np.zeros(len(df))
    Tk = df["Tset"].to_numpy() + 273.15 if v.get("basis", "Tset") == "Tset" else np.exp(df[fcol].to_numpy())
    w = -(lng + 3.0 * df["ln_H"].to_numpy())                       # ln Fo（シフト前の γ̇ で計算。定数は省く）
    return np.column_stack([lng, w]), np.column_stack([lng + shift_ln(Tk, v["Eh"]), w]), shift_ln(Tk, v["Ev"])


def fit_variant(v, tr, mode, restarts):
    X0, Xs_, ysh = design(tr, v, mode)
    mu_x, sd_x = X0.mean(axis=0), X0.std(axis=0)                   # E に依存しない定数（シフト前の入力の平均・標準偏差）
    y = tr["ln_eta"].to_numpy()
    y0 = y.mean()                                                   # E に依存しない固定の定数
    gp = GaussianProcessRegressor(kernel=kernel_for(X0.shape[1], True), n_restarts_optimizer=restarts, random_state=SEED)
    gp.fit((Xs_ - mu_x) / sd_x, y - ysh - y0)
    return {"gp": gp, "mu_x": mu_x, "sd_x": sd_x, "y0": y0, "v": v, "mode": mode}


def predict_variant(fit, df):
    """予測平均、予測標準偏差（ノイズ込み）、潜在関数の標準偏差、ノイズの標準偏差"""
    X0, Xs_, ysh = design(df, fit["v"], fit["mode"])
    mu, sd = fit["gp"].predict((Xs_ - fit["mu_x"]) / fit["sd_x"], return_std=True)
    noise = fit["gp"].kernel_.k2.noise_level
    return mu + fit["y0"] + ysh, sd, np.sqrt(np.maximum(sd ** 2 - noise, 0.0)), float(np.sqrt(noise))


def cond_table(df):
    """条件ごとの平均（ln η と ln γ̇ と ln Tff）。尤度の比較に使う"""
    g = df.groupby("cond", sort=False)
    t = g.agg(Tset=("Tset", "first"), H=("H", "first"), v=("v", "first"), ln_eta=("ln_eta", "mean"), ln_gd=("ln_gd", "mean"),
              ln_Tff=("ln_Tff", "mean"), n=("ln_eta", "size"), is_sens=("is_sens", "max")).reset_index()
    t["ln_H"], t["ln_gd_cm"], t["ln_Tff_cm"] = np.log(t["H"]), t["ln_gd"], t["ln_Tff"]
    return t


STRUCT = {   # 1 次元の構造：E → 構造の指定
    "H": lambda e: {"kind": "shift", "Eh": e, "Ev": e, "basis": "Tset"},
    "A": lambda e: {"kind": "shift", "Eh": 0.0, "Ev": e, "basis": "Tset"},
    "Hz": lambda e: {"kind": "shift", "Eh": e, "Ev": 0.0, "basis": "Tset"},
    "F": lambda e: {"kind": "shift", "Eh": e, "Ev": e, "basis": "Tff"},
}
STRUCT_JA = {"H": "H：横・縦を同じ E（当初の §5）", "A": "A：縦だけシフト", "Hz": "Hz：横だけシフト", "B": "B：横・縦を別々", "F": "F：Tff でシフト（参考）",
             "N": "N：温度なし（E = 0）", "H40": "H40：E = 40 kJ/mol", "C": "C：通常の GPR［γ̇, H, Tset］"}
STRUCT_COL = {"H": "tab:red", "A": "tab:blue", "Hz": "tab:green", "B": "tab:purple", "F": "tab:orange", "N": "gray", "H40": "tab:brown", "C": "k"}
print("構造:", list(STRUCT_JA))

## 4. プロファイル尤度（条件平均）

各構造について、E の格子の各点で GPR を学習し、対数周辺尤度（LML）を求める。**LML が大きいほど、データに合っている**。学習データは 3 通り：

- **S_V**：5data（190・210・230 ℃、93 条件）
- **S_T**：5data の 190・230 ℃（210 ℃ を除く）
- **ALL**：5data ＋ 3data（129 条件。最終的な E の推定値用）

LML の最大値から 1.92 以内（尤度比検定の 95% 区間にあたる）の E の範囲も求める。B は 2 次元の格子（E_h × E_v）。

In [ ]:
# セル4：プロファイルの計算（並列。数分かかる）。まず粗い格子で全体を見て、最良の点の近くを細かい格子で計算し直す
def prof_job(cdf, v):
    warnings.simplefilter("ignore")                  # 並列の子プロセスでも、下限への張り付きの警告を出さない（kernels.csv に記録する）
    f = fit_variant(v, cdf, "cm", N_RESTARTS_PROFILE)
    return float(f["gp"].log_marginal_likelihood_value_), float(np.sqrt(f["gp"].kernel_.k2.noise_level))


def profile_sets(mat):
    tr_v, _ = split_V(mat, MAIN_DATA)
    tr_t, _, _ = split_T(mat, MAIN_DATA, TEMP_HOLDOUT)
    return {"S_V": tr_v, "S_T": tr_t, "ALL": pool_of(mat, MAIN_DATA)}


def grid_jobs(cdf, mat, sname, grid_1d, structs, eh_grid=(), ev_grid=()):
    """計算の仕事の一覧 [(条件平均のデータ, 構造の指定, (材料, 学習データ, 構造, E, E_h, E_v))]。構造 B は (E_h, E_v) の格子"""
    jl = []
    for st in structs:
        for e in grid_1d:
            e = float(e)
            jl.append((cdf, STRUCT[st](e), (mat, sname, st, e, e if st in ("H", "Hz", "F") else 0.0, e if st in ("H", "A", "F") else 0.0)))
    for eh in eh_grid:
        for ev in ev_grid:
            jl.append((cdf, {"kind": "shift", "Eh": float(eh), "Ev": float(ev), "basis": "Tset"}, (mat, sname, "B", np.nan, float(eh), float(ev))))
    return jl


def run_profile(jl, stage):
    out = Parallel(n_jobs=N_JOBS, batch_size=4)(delayed(prof_job)(c, v) for c, v, _ in jl)
    return pd.DataFrame([{"material": MATS[k[0]], "mat": k[0], "set": k[1], "structure": k[2], "E": k[3], "Eh": k[4], "Ev": k[5], "LML": o[0], "noise_sd": o[1], "stage": stage}
                         for (_, _, k), o in zip(jl, out)])


def best_table(pdf):
    """構造ごとの最良の E（B は E_h と E_v）と、LML が最大から 1.92 以内の範囲"""
    rows = []
    for (mat, sname, st), g in pdf.groupby(["mat", "set", "structure"]):
        r = g.loc[g["LML"].idxmax()]
        h = pdf[(pdf["mat"] == mat) & (pdf["set"] == sname) & (pdf["structure"] == "H") & (pdf["E"] == 0.0)]
        n0 = h["LML"].iloc[0]                                                                 # 温度なし（E = 0）の LML
        inside = g[g["LML"] >= r["LML"] - 1.92]
        row = {"material": MATS[mat], "mat": mat, "set": sname, "structure": st, "LML_best": r["LML"], "LML_at_E0": n0, "dLML_vs_E0": r["LML"] - n0}
        if st == "B":
            row.update({"Eh_best": r["Eh"], "Ev_best": r["Ev"], "n_in_95": len(inside), "Eh_95_lo": inside["Eh"].min(), "Eh_95_hi": inside["Eh"].max(),
                        "Ev_95_lo": inside["Ev"].min(), "Ev_95_hi": inside["Ev"].max()})
        else:
            g40 = g[np.isclose(g["E"], E_LIT)]
            row.update({"E_best": r["E"], "E_95_lo": inside["E"].min(), "E_95_hi": inside["E"].max(), "at_grid_edge": bool(r["E"] in (g["E"].min(), g["E"].max())),
                        "LML_at_E40": g40["LML"].iloc[0] if len(g40) else np.nan})
        rows.append(row)
    return pd.DataFrame(rows)


def refine(coarse, sets_cdf):
    """粗い格子で最良の点の近くを、細かい格子（1 次元は ±2.5 kJ/mol を 0.5 刻み、B は ±5 kJ/mol を 1 刻み）で計算し直して、まとめる"""
    best0, jl = best_table(coarse), []
    for _, r in best0.iterrows():
        cdf = sets_cdf[(r["mat"], r["set"])]
        if r["structure"] == "B":
            jl += grid_jobs(cdf, r["mat"], r["set"], [], [], eh_grid=r["Eh_best"] + np.arange(-5.0, 5.01, 1.0), ev_grid=r["Ev_best"] + np.arange(-5.0, 5.01, 1.0))
        else:
            jl += grid_jobs(cdf, r["mat"], r["set"], r["E_best"] + np.arange(-2.5, 2.51, 0.5), [r["structure"]])
    fine = run_profile(jl, "細かい格子")
    allp = pd.concat([coarse, fine], ignore_index=True)
    return allp.drop_duplicates(subset=["mat", "set", "structure", "Eh", "Ev"], keep="first").reset_index(drop=True)


SETS = {(mat, sname): cond_table(tr) for mat in MATS for sname, tr in profile_sets(mat).items()}
t0 = time.time()
jl = [j for (mat, sname), cdf in SETS.items() for j in grid_jobs(cdf, mat, sname, E_GRID, ["H", "A", "Hz", "F"], EH_GRID, EV_GRID)]
coarse = run_profile(jl, "粗い格子")
print(f"粗い格子：{len(jl)} 点、{(time.time() - t0) / 60:.1f} 分")
prof_df = refine(coarse, SETS)
print(f"細かい格子を足して、全部で {len(prof_df)} 点、{(time.time() - t0) / 60:.1f} 分")
prof_df.drop(columns="mat").to_csv(os.path.join(OUT_DIR, "profile_likelihood.csv"), index=False, encoding="utf-8-sig")

In [ ]:
# セル5：最良の E と、その区間
best_df = best_table(prof_df)
best_df.drop(columns="mat").to_csv(os.path.join(OUT_DIR, "best_E.csv"), index=False, encoding="utf-8-sig")
print("【最良の E（kJ/mol）と LML】E_95_lo〜hi：LML が最大から 1.92 以内の範囲。dLML_vs_E0：温度なし（E = 0）との LML の差（正なら、シフトのほうがデータに合う）")
print(best_df.drop(columns="mat").round(2).to_string(index=False))

## 5. 感度確認：0.5 mm・20 mm/s・190 ℃（P02）を除いた場合

問題データ一覧の P02 は、他の温度と傾向が逆になっている条件。学習から除いて、H と A の最良の E がどれだけ動くかを見る（S_V の学習）。**粗い格子 1 目（2.5 kJ/mol）以上動いたら、影響が大きい**と判断する。

In [ ]:
# セル6：P02 を除いたプロファイル
SETS2 = {(mat, "S_V（P02 を除く）"): cond_table(split_V(mat, MAIN_DATA)[0].pipe(lambda t: t[~t["is_sens"]])) for mat in MATS}
jl2 = [j for (mat, sname), cdf in SETS2.items() for j in grid_jobs(cdf, mat, sname, E_GRID, ["H", "A"])]
sens_prof = refine(run_profile(jl2, "粗い格子"), SETS2)
best2 = best_table(sens_prof)
srows = []
for _, r in best2.iterrows():
    e_all = float(best_df[(best_df["mat"] == r["mat"]) & (best_df["set"] == "S_V") & (best_df["structure"] == r["structure"])]["E_best"].iloc[0])
    srows.append({"material": r["material"], "structure": r["structure"], "E_best_すべて": e_all, "E_best_P02を除く": float(r["E_best"]), "動き [kJ/mol]": float(r["E_best"]) - e_all,
                  "影響が大きい": abs(float(r["E_best"]) - e_all) >= 2.5 - 1e-9})
sens_df = pd.DataFrame(srows)
sens_prof.drop(columns="mat").to_csv(os.path.join(OUT_DIR, "profile_likelihood_P02_excluded.csv"), index=False, encoding="utf-8-sig")
sens_df.to_csv(os.path.join(OUT_DIR, "sensitivity_E.csv"), index=False, encoding="utf-8-sig")
print(sens_df.round(2).to_string(index=False))
for mat in MATS:
    s = sens_df[sens_df["material"] == MATS[mat]]
    judge(not s["影響が大きい"].any(), f"{MATS[mat]}：P02 を除いても、H・A の最良の E が 2.5 kJ/mol（粗い格子 1 目）以上は動かない（動き {s['動き [kJ/mol]'].round(1).tolist()}）。動いたなら、P02 だけを除いたファイルを作る")

## 6. 予測の評価（S_V・S_T）

4 の最良の E（その学習データのもの）を使い、ショット単位の GPR を学習して、検証データを予測する。S_V の学習には S_V の E、S_T の学習には S_T の E を使う（検証データの情報は使わない）。

- 構造：C・N・H・H40・A・Hz・B・F（上の表）
- 入力の γ̇：`cm`（条件平均。成形条件から見た予測）と `shot`（ショットごと）
- 評価：S_V（低速側・高速側）、S_T（5data の 210 ℃）、S_T3（3data の 210 ℃：速度の外挿も重なる）。「（感度確認の条件を除く）」は、検証の P03（PP の 190 ℃・0.5 mm・320 mm/s）を評価から除いた指標。

In [ ]:
# セル7：最良の E で学習し直して、予測を評価する
def e_of(mat, sname, st):
    r = best_df[(best_df["mat"] == mat) & (best_df["set"] == sname) & (best_df["structure"] == st)].iloc[0]
    return r


def variant_defs(mat, sname):
    eH, eA, eZ, eF, rB = e_of(mat, sname, "H")["E_best"], e_of(mat, sname, "A")["E_best"], e_of(mat, sname, "Hz")["E_best"], e_of(mat, sname, "F")["E_best"], e_of(mat, sname, "B")
    return {"C": {"kind": "free"}, "N": STRUCT["H"](0.0), "H": STRUCT["H"](float(eH)), "H40": STRUCT["H"](E_LIT), "A": STRUCT["A"](float(eA)),
            "Hz": STRUCT["Hz"](float(eZ)), "B": {"kind": "shift", "Eh": float(rB["Eh_best"]), "Ev": float(rB["Ev_best"]), "basis": "Tset"}, "F": STRUCT["F"](float(eF))}


def fit_job(tr, v, mode):
    warnings.simplefilter("ignore")
    return fit_variant(v, tr, mode, N_RESTARTS_FINAL)


tasks, tkeys = [], []
for mat in MATS:
    for split in ["S_V", "S_T"]:
        tr = split_V(mat, MAIN_DATA)[0] if split == "S_V" else split_T(mat, MAIN_DATA, TEMP_HOLDOUT)[0]
        for name, v in variant_defs(mat, split).items():
            for mode in ["cm", "shot"]:
                tasks.append((tr, v, mode))
                tkeys.append((mat, split, name, mode))
t0 = time.time()
fits = Parallel(n_jobs=N_JOBS)(delayed(fit_job)(*t) for t in tasks)
FIT5 = dict(zip(tkeys, fits))
print(f"{len(fits)} 個のモデルを学習。{(time.time() - t0) / 60:.1f} 分")

rows, krows = [], []
for (mat, split, name, mode), fit in FIT5.items():
    tests = {"S_V": {"S_V": split_V(mat, MAIN_DATA)[1]}, "S_T": {"S_T": split_T(mat, MAIN_DATA, TEMP_HOLDOUT)[1], "S_T3": split_T(mat, MAIN_DATA, TEMP_HOLDOUT)[2]}}[split]
    for tname, te in tests.items():
        subsets = {"全体": np.ones(len(te), bool)}
        for g in te["group"].unique():
            if g != "全体":
                subsets[g] = (te["group"] == g).to_numpy()
        if te["is_sens"].any():
            subsets.update({s + SENS_SUFFIX: m & ~te["is_sens"].to_numpy() for s, m in list(subsets.items())})
        mu, sd, sd_f, noise_sd = predict_variant(fit, te)
        y = te["ln_eta"].to_numpy()
        for sub, mask in subsets.items():
            if mask.sum():
                rows.append({"material": MATS[mat], "split": tname, "mode": mode, "variant": name, "subset": sub, "n_test": int(mask.sum()), **evaluate(y[mask], mu[mask], sd[mask]), "noise_sd": noise_sd})
    v = fit["v"]
    k = fit["gp"].kernel_
    krows.append({"material": MATS[mat], "split": split, "mode": mode, "variant": name, "Eh": v.get("Eh", np.nan), "Ev": v.get("Ev", np.nan), "LML": float(fit["gp"].log_marginal_likelihood_value_),
                  "noise_sd": float(np.sqrt(k.k2.noise_level)), "noise_at_lower_bound": bool(k.k2.noise_level < NOISE_LB * 1.05)})
metrics5 = pd.DataFrame(rows)
kern5 = pd.DataFrame(krows)
metrics5.to_csv(os.path.join(OUT_DIR, "metrics.csv"), index=False, encoding="utf-8-sig")
kern5.to_csv(os.path.join(OUT_DIR, "kernels.csv"), index=False, encoding="utf-8-sig")
VARS = ["C", "N", "H", "H40", "A", "Hz", "B", "F"]


def m5(mat, split, variant, sub, col="RMSE", mode="cm"):
    r = metrics5[(metrics5["material"] == MATS[mat]) & (metrics5["split"] == split) & (metrics5["mode"] == mode) & (metrics5["variant"] == variant) & (metrics5["subset"] == sub)]
    return float(r[col].iloc[0]) if len(r) else np.nan


for mode in ["cm", "shot"]:
    for mat in MATS:
        print(f"\n【{MATS[mat]}】入力の γ̇ = {'条件平均（cm）' if mode == 'cm' else 'ショットごと（shot）'}。RMSE（ln η）／NLPD")
        cols = [("S_V", "低速側"), ("S_V", "高速側"), ("S_T", "全体"), ("S_T3", "全体")]
        t = pd.DataFrame({f"{s}{'' if sub == '全体' else ' ' + sub}": {v: f"{m5(mat, s, v, sub, 'RMSE', mode):.3f}／{m5(mat, s, v, sub, 'NLPD', mode):.2f}" for v in VARS} for s, sub in cols})
        print(t.to_string())

## 7. 設定温度の効果

実測と、各構造の予測を比べる（3b の図5と同じ見方）。

- **実測の比**：同じ厚み・同じ γ̇ で、190 ℃ の粘度 ÷ 230 ℃ の粘度（感度確認の条件は除く）。
- **モデルの比**：5data＋3data の 210 ℃ のショットについて、ほかの入力はそのまま、Tset だけを 190 ℃・230 ℃ にして予測した粘度の比（S_V の学習、cm）。N と F は Tset に反応しないので、比が 1 になる。

In [ ]:
# セル8：設定温度の効果
H_LEVELS = [0.5, 1.0, 1.5, 2.0]


def observed_ratio(mat):
    d = pool_of(mat, MAIN_DATA)
    d = d[~d["is_sens"]]
    cm = d.groupby(["Tset", "H", "v"]).agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean")).reset_index()
    rows = []
    for h in H_LEVELS:
        a, c = cm[(cm["H"] == h) & (cm["Tset"] == 190)], cm[(cm["H"] == h) & (cm["Tset"] == 230)].sort_values("lg")
        ins = ((a["lg"] >= c["lg"].min()) & (a["lg"] <= c["lg"].max())).to_numpy()
        r = np.exp(a["le"].to_numpy()[ins] - np.interp(a["lg"].to_numpy()[ins], c["lg"], c["le"]))
        rows.append({"material": MATS[mat], "H": h, "observed": float(np.median(r)), "n": int(ins.sum())})
    return pd.DataFrame(rows)


def model_ratio(split="S_V", mode="cm"):
    rows = []
    for mat in MATS:
        base = pool_of(mat, MAIN_DATA)
        base = base[(base["Tset"] == 210) & ~base["is_sens"]]
        for name in VARS:
            fit = FIT5[(mat, split, name, mode)]
            lo, hi = base.copy(), base.copy()
            lo["Tset"], hi["Tset"] = 190.0, 230.0
            r = np.exp(predict_variant(fit, lo)[0] - predict_variant(fit, hi)[0])
            for h, g in base.assign(r=r).groupby("H"):
                rows.append({"material": MATS[mat], "split": split, "mode": mode, "variant": name, "H": h, "ratio_190_230": float(np.median(g["r"]))})
    return pd.DataFrame(rows)


obs5 = pd.concat([observed_ratio(m) for m in MATS], ignore_index=True)
eff5 = pd.concat([model_ratio("S_V"), model_ratio("S_T")], ignore_index=True)
obs5.to_csv(os.path.join(OUT_DIR, "observed_ratio.csv"), index=False, encoding="utf-8-sig")
eff5.to_csv(os.path.join(OUT_DIR, "tset_effect.csv"), index=False, encoding="utf-8-sig")
print("実測の 190/230 ℃ の粘度比（同じ γ̇、厚みごと）")
print(obs5.pivot(index="material", columns="H", values="observed").round(3).to_string())
for sp in ["S_V", "S_T"]:
    print(f"\nモデルの比（{sp} の学習、Tset だけを 190 → 230 ℃）")
    print(eff5[eff5["split"] == sp].pivot_table(index=["material", "variant"], columns="H", values="ratio_190_230").reindex(VARS, level=1).round(3).to_string())

## 8. 本命と対案の比較（ユーザー決定、2026-10-12）

本命（`MAIN_STRUCT`、既定は B）と対案（`ALT_STRUCT`、既定は A）を、基準 C（構造なしの通常の GPR）と並べて比べる。後で自分で実行して、どちらを使うか判断するための節。

- **E**：S_V・S_T・ALL の学習での最良の E（E_h・E_v。A は E_h = 0、H は E_h = E_v）。学習データを変えても動かないか。
- **LML**：S_V の学習での、最良の E の対数周辺尤度と、温度なし（E = 0）との差。**大きいほどデータに合う**。差が 2 以上なら、はっきりした違い。
- **検証の指標**：RMSE（ln η）・95% 被覆率・NLPD。入力の γ̇ は条件平均（cm）。S_V の低速側・高速側、S_T（210 ℃）、S_T3（3data の 210 ℃）。
- **設定温度の効果**：190 ℃ と 230 ℃ の粘度の比（同じ γ̇。厚みごと）と、実測。

結果は `main_vs_alt.csv` に保存する。

In [ ]:
# セル13.5：本命と対案の比較
PANELS5 = [("S_V", "低速側", "S_V_low"), ("S_V", "高速側", "S_V_high"), ("S_T", "全体", "S_T"), ("S_T3", "全体", "S_T3")]
ROLES = [("本命", MAIN_STRUCT), ("対案", ALT_STRUCT), ("基準", "C")]
cmp_rows = []
for mat in MATS:
    for role, name in ROLES:
        row = {"material": MATS[mat], "role": role, "structure": name}
        for sname in ["S_V", "S_T", "ALL"]:
            v = variant_defs(mat, sname)[name]
            row[f"Eh_{sname}"], row[f"Ev_{sname}"] = v.get("Eh", np.nan), v.get("Ev", np.nan)
        b = best_df[(best_df["mat"] == mat) & (best_df["set"] == "S_V") & (best_df["structure"] == name)]
        row["LML_S_V"], row["dLML_vs_E0_S_V"] = (float(b["LML_best"].iloc[0]), float(b["dLML_vs_E0"].iloc[0])) if len(b) else (np.nan, np.nan)
        for split, sub, lab in PANELS5:
            for metric in ["RMSE", "coverage95", "NLPD"]:
                row[f"{metric}_{lab}"] = m5(mat, split, name, sub, metric)
        e = eff5[(eff5["split"] == "S_V") & (eff5["material"] == MATS[mat]) & (eff5["variant"] == name)].sort_values("H")
        for h, r in zip(e["H"], e["ratio_190_230"]):
            row[f"ratio_{h}"] = float(r)
        cmp_rows.append(row)
    o = obs5[obs5["material"] == MATS[mat]].sort_values("H")
    cmp_rows.append({"material": MATS[mat], "role": "実測", "structure": "―", **{f"ratio_{h}": float(r) for h, r in zip(o["H"], o["observed"])}})
cmp_df = pd.DataFrame(cmp_rows)
cmp_df.to_csv(os.path.join(OUT_DIR, "main_vs_alt.csv"), index=False, encoding="utf-8-sig")
RATIO_COLS = [f"ratio_{h}" for h in H_LEVELS]
for mat in MATS:
    t = cmp_df[cmp_df["material"] == MATS[mat]].set_index("role")
    print(f"\n【{MATS[mat]}】本命 {MAIN_STRUCT}、対案 {ALT_STRUCT}、基準 C")
    print("E [kJ/mol]（E_h／E_v。S_V・S_T・ALL の学習）")
    print(t.loc[[r for r, _ in ROLES], [f"{p}_{s}" for s in ["S_V", "S_T", "ALL"] for p in ["Eh", "Ev"]]].round(1).to_string())
    print("LML（S_V の学習）と、検証の RMSE")
    print(t.loc[[r for r, _ in ROLES], ["LML_S_V", "dLML_vs_E0_S_V"] + [f"RMSE_{lab}" for _, _, lab in PANELS5]].round(3).to_string())
    print("190/230 ℃ の粘度の比（同じ γ̇。厚みごと。実測を含む）")
    print(t.loc[[r for r, _ in ROLES] + ["実測"], RATIO_COLS].round(3).to_string())
for mat in MATS:
    n = MATS[mat]
    t = cmp_df[cmp_df["material"] == n].set_index("role")
    lm, la = t.loc["本命", "LML_S_V"], t.loc["対案", "LML_S_V"]
    if np.isnan(lm) or np.isnan(la):
        print(f"（{n}：本命か対案が N・H40・C なので、LML は比べない）")
    else:
        judge(lm - la >= 2.0, f"{n}：本命 {MAIN_STRUCT} の LML（{lm:.1f}）が、対案 {ALT_STRUCT}（{la:.1f}）より 2 以上大きい（差 {lm - la:.1f}）→ 本命のほうがデータに合う")
    bad = [lab for _, _, lab in PANELS5 if t.loc["本命", f"RMSE_{lab}"] > 1.05 * t.loc["対案", f"RMSE_{lab}"]]
    judge(not bad, f"{n}：本命の検証の RMSE が、対案の 1.05 倍以内（超えた検証: {bad or 'なし'}）。4 つの検証：" + "、".join(f"{lab} {t.loc['本命', f'RMSE_{lab}']:.3f}／{t.loc['対案', f'RMSE_{lab}']:.3f}" for _, _, lab in PANELS5))
    bad = [lab for _, _, lab in PANELS5 if t.loc["本命", f"RMSE_{lab}"] > 1.1 * t.loc["基準", f"RMSE_{lab}"]]
    judge(not bad, f"{n}：本命の検証の RMSE が、基準 C の 1.1 倍以内（超えた検証: {bad or 'なし'}。同等なら、構造がデータで支持されている）")
    dev = {r: float(np.abs(t.loc[r, RATIO_COLS].astype(float).to_numpy() - t.loc["実測", RATIO_COLS].astype(float).to_numpy()).max()) for r in ["本命", "対案"]}
    judge(dev["本命"] <= dev["対案"], f"{n}：190/230 ℃ の粘度比が、本命のほうが実測に近い（実測との差の最大：本命 {dev['本命']:.3f}、対案 {dev['対案']:.3f}）。対案が A のように厚みによらず一定だと、厚みごとの違いを表せない")
    for role in ["本命", "対案"]:
        spread = {c: float(t.loc[role, [f"{c}_S_V", f"{c}_S_T", f"{c}_ALL"]].astype(float).max() - t.loc[role, [f"{c}_S_V", f"{c}_S_T", f"{c}_ALL"]].astype(float).min()) for c in ["Eh", "Ev"]}
        if not any(np.isnan(v) for v in spread.values()):
            judge(max(spread.values()) <= 5.0, f"{n}：{role}（{t.loc[role, 'structure']}）の E が、学習データ（S_V・S_T・ALL）を変えても 5 kJ/mol 以内（E_h の動き {spread['Eh']:.1f}、E_v の動き {spread['Ev']:.1f}）")

## 9. 図

- 図0：構造の考え方の模式図（実データではない）
- 図1：プロファイル尤度（E と LML の関係）。構造ごと・学習データごと
- 図2：B の 2 次元プロファイル（E_h × E_v）。材料ごとに 1 枚（2a：PP、2b：粉砕材混合 PP）
- 図3：マスターカーブ（換算後のデータ）。重なっていれば、その構造でシフトできている
- 図4：設定温度ごとの予測曲線（構造ごと）
- 図5：評価指標の比較
- 図6：設定温度の効果（190/230 ℃ の粘度比）

In [ ]:
# セル8.9：図0 構造の考え方（模式図。実データではない）
E_DEMO = {"H": (30.0, 30.0), "A": (0.0, 30.0), "Hz": (30.0, 0.0), "B": (15.0, 30.0)}      # (E_h, E_v) [kJ/mol]。形が見やすいように、実測より大きな値にした例
T0_COL = {190: "tab:blue", 210: "tab:green", 230: "tab:red"}
TK0 = {190: 463.15, 210: 483.15, 230: 503.15}
ug = np.linspace(np.log(3.0), np.log(600000.0), 1500)                                      # ln γ̇（基準の 210 ℃ の曲線を描く格子）
slope0 = -0.4 - 1.5 / (1.0 + np.exp(-(ug - np.log(3000.0)) / 1.2))                         # 基準の曲線の傾き：低せん断で −0.4、高せん断で −1.9（見かけの流動曲線は、せん断速度とともに急になる）
f_ref = np.r_[0.0, np.cumsum((slope0[1:] + slope0[:-1]) / 2 * np.diff(ug))] + np.log(2000.0)    # ln η の基準の曲線（任意の単位）
xs0 = np.linspace(np.log(30.0), np.log(60000.0), 300)
fig, axes = plt.subplots(2, len(E_DEMO), figsize=(5.0 * len(E_DEMO), 9.5), sharex=True)
for j, (name, (eh, ev)) in enumerate(E_DEMO.items()):
    curves0 = {t: shift_ln(TK0[t], ev) + np.interp(xs0 + shift_ln(TK0[t], eh), ug, f_ref) for t in TK0}          # ln η = ln a(E_v) + f(ln γ̇ + ln a(E_h))
    for t, col in T0_COL.items():
        axes[0, j].plot(np.exp(xs0), np.exp(curves0[t]), color=col, lw=2, label=f"{t} ℃")
    axes[0, j].set_xscale("log")
    axes[0, j].set_yscale("log")
    axes[0, j].set_title(f"{STRUCT_JA[name]}\nE_h = {eh:g}、E_v = {ev:g} kJ/mol", fontsize=10)
    axes[0, j].grid(alpha=0.3, which="both")
    axes[1, j].plot(np.exp(xs0), np.exp(curves0[190] - curves0[230]), color="k", lw=2)
    axes[1, j].axhline(1.0, color="gray", lw=0.8, ls="--")
    axes[1, j].set_xscale("log")
    axes[1, j].set_yscale("log")
    axes[1, j].set_ylim(0.25, 3.5)
    axes[1, j].set_yticks([0.3, 0.5, 0.7, 1, 1.5, 2, 3])
    axes[1, j].set_yticklabels(["0.3", "0.5", "0.7", "1", "1.5", "2", "3"])
    axes[1, j].yaxis.set_minor_formatter(plt.NullFormatter())
    axes[1, j].set_xlabel(r"せん断速度 $\dot{\gamma}$ [1/s]")
    axes[1, j].grid(alpha=0.3, which="both")
axes[0, 0].set_ylabel(r"粘度 $\eta$（任意の単位）")
axes[1, 0].set_ylabel("粘度の比（190 ℃ ÷ 230 ℃）")
axes[0, 0].legend(fontsize=9)
fig.suptitle("構造の考え方の模式図（実データではない。E は、形が見やすいように大きめの例）。上段：温度ごとの流動曲線、下段：同じ γ̇ での粘度の比（190 ℃ ÷ 230 ℃）\n"
             "H は、傾きが −1 より急な所で比が 1 を下回る（向きが逆になる）。A は比が一定。B は、A の一定の比に、場所で変わる分が加わる", fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(os.path.join(OUT_DIR, "fig0_shift_concept.png"), dpi=FIG_DPI)
plt.show()

In [ ]:
# セル9：図1 プロファイル尤度（1 次元）
SET_JA = {"S_V": "S_V の学習（5data）", "S_T": "S_T の学習（190・230 ℃）", "ALL": "ALL（5data＋3data）"}
fig, axes = plt.subplots(len(MATS), 3, figsize=(17, 9), sharex=True)
for i, mat in enumerate(MATS):
    for j, sname in enumerate(["S_V", "S_T", "ALL"]):
        ax = axes[i, j]
        for st in ["H", "A", "Hz", "F"]:
            g = prof_df[(prof_df["mat"] == mat) & (prof_df["set"] == sname) & (prof_df["structure"] == st)].sort_values("E")
            ax.plot(g["E"], g["LML"], "-o", ms=2.5, lw=1.2, color=STRUCT_COL[st], label=STRUCT_JA[st])
            b = g.loc[g["LML"].idxmax()]
            ax.plot([b["E"]], [b["LML"]], "*", ms=14, color=STRUCT_COL[st], mec="k")
        ax.axvline(0, color="gray", lw=0.8)
        ax.axvline(E_LIT, color="k", ls=":", lw=1, label="文献値の目安 40 kJ/mol")
        ax.set_title(f"{MATS[mat]}：{SET_JA[sname]}", fontsize=10)
        ax.set_xlabel("E [kJ/mol]")
        ax.set_ylabel("対数周辺尤度（大きいほど良い）")
        ax.grid(alpha=0.3)
        top = prof_df[(prof_df["mat"] == mat) & (prof_df["set"] == sname) & prof_df["structure"].isin(["H", "A", "Hz", "F"])]["LML"].max()
        ax.set_ylim(top - 130, top + 10)
        if i == 0 and j == 0:
            ax.legend(fontsize=7, loc="lower right")
fig.suptitle("プロファイル尤度（条件平均）：★ が最良の E。E = 0 が温度なし。A・Hz は鋭い山、H・F は平らで、折れ線がでこぼこ（最適化が局所解に止まった点を含む）。B は図2", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig1_profile_likelihood.png"), dpi=FIG_DPI)
plt.show()

In [ ]:
# セル10：図2 B の 2 次元プロファイル（材料ごとに 1 枚。上段：粗い格子の全体、下段：最良の点の近くの細かい格子）
for mat in MATS:
    fig, axes = plt.subplots(2, 3, figsize=(17, 9.5))
    for j, sname in enumerate(["S_V", "S_T", "ALL"]):
        g = prof_df[(prof_df["mat"] == mat) & (prof_df["set"] == sname) & (prof_df["structure"] == "B")]
        b = g.loc[g["LML"].idxmax()]
        top = b["LML"]
        for i, (sub, ttl) in enumerate([(g[g["stage"] == "粗い格子"], "全体（粗い格子。5 kJ/mol 刻み）"),
                                         (g[(abs(g["Eh"] - b["Eh"]) <= 5.001) & (abs(g["Ev"] - b["Ev"]) <= 5.001)], "最良の点の近く（細かい格子。1 kJ/mol 刻み）")]):
            ax = axes[i, j]
            Z = sub.pivot(index="Ev", columns="Eh", values="LML")
            im = ax.pcolormesh(Z.columns, Z.index, np.clip(Z.to_numpy(), top - 40, top), shading="nearest", cmap="viridis")
            if i == 1:
                ax.contour(Z.columns, Z.index, Z.to_numpy(), levels=[top - 1.92], colors="w", linewidths=1.2)
            ax.plot([b["Eh"]], [b["Ev"]], "*", ms=14, color="r", mec="w", label=f"最良（E_h = {b['Eh']:.0f}、E_v = {b['Ev']:.0f}）")
            lim = [min(Z.columns.min(), Z.index.min()), max(Z.columns.max(), Z.index.max())]
            ax.plot(lim, lim, "w--", lw=0.8, label="E_h = E_v（H）")
            ax.axvline(0, color="w", lw=0.8, ls=":")
            ax.axhline(0, color="w", lw=0.8, ls=":")
            ax.set_xlim(Z.columns.min(), Z.columns.max())
            ax.set_ylim(Z.index.min(), Z.index.max())
            ax.set_title(f"{MATS[mat]}：{SET_JA[sname]}\n{ttl}", fontsize=9)
            ax.set_xlabel("横方向 E_h [kJ/mol]")
            ax.set_ylabel("縦方向 E_v [kJ/mol]")
            ax.legend(fontsize=7, loc="upper left")
            fig.colorbar(im, ax=ax, label="対数周辺尤度")
    fig.suptitle(f"{MATS[mat]}：B（横・縦を別々）の対数周辺尤度。白の輪郭：最大から 1.92 以内。破線：H（E_h = E_v）、点線：A（E_h = 0）・Hz（E_v = 0）", fontsize=11)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"fig2{'a' if mat == 'PP' else 'b'}_profile_2d_B_{mat}.png"), dpi=FIG_DPI)
    plt.show()

In [ ]:
# セル11：図3 マスターカーブ（ALL の学習データを、構造ごとの最良の E で換算）
T_COL = {190: "tab:blue", 210: "tab:green", 230: "tab:red"}
MASTER = [("N", "N：温度なし"), ("H", "H：横・縦同じ E"), ("A", "A：縦だけ"), ("B", "B：横・縦別々")]
for mat in MATS:
    pool = pool_of(mat, MAIN_DATA)
    cdf = cond_table(pool)
    defs = variant_defs(mat, "ALL")
    fig, axes = plt.subplots(4, len(MASTER), figsize=(5.0 * len(MASTER), 14), sharey="row")
    for j, (name, lab) in enumerate(MASTER):
        v = defs[name]
        X0, Xs_, ysh = design(cdf, v, "cm")
        for i, h in enumerate(H_LEVELS):
            ax = axes[i, j]
            for t, col in T_COL.items():
                m = ((cdf["H"] == h) & (cdf["Tset"] == t)).to_numpy()
                ax.scatter(Xs_[m, 0], cdf["ln_eta"].to_numpy()[m] - ysh[m], s=16, color=col, alpha=0.8, label=f"{t} ℃")
            ax.grid(alpha=0.3)
            if j == 0:
                ax.set_ylabel(f"厚み {h} mm\n" + r"換算後の $\ln\eta$（縦シフトを引いた値）")
            if i == 0:
                ax.set_title(f"{lab}\n(E_h, E_v) = ({v['Eh']:.1f}, {v['Ev']:.1f}) kJ/mol", fontsize=10)
            if i == 3:
                ax.set_xlabel(r"換算後の $\ln\dot{\gamma}$ ＋ 横シフト")
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(f"{MATS[mat]}：マスターカーブ（条件平均。3 つの温度の点が重なっていれば、その構造でシフトできている）", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    fig.savefig(os.path.join(OUT_DIR, f"fig3{'a' if mat == 'PP' else 'b'}_master_curve_{mat}.png"), dpi=FIG_DPI)
    plt.show()

In [ ]:
# セル12：図4 設定温度ごとの予測曲線（S_V の学習、cm。列が構造、行が厚み）
CURVE_VARS = ["C", "N", "A", "B", "H"]
for mat in MATS:
    allp = pool_of(mat, MAIN_DATA)
    te_cond = set(D[(mat, "3data", MAIN_DATA)]["cond"])
    fig, axes = plt.subplots(4, len(CURVE_VARS), figsize=(4.6 * len(CURVE_VARS), 14), sharex="row", sharey="row")
    for j, name in enumerate(CURVE_VARS):
        fit = FIT5[(mat, "S_V", name, "cm")]
        for i, h in enumerate(H_LEVELS):
            ax = axes[i, j]
            for t, col in T_COL.items():
                sub = allp[(allp["H"] == h) & (allp["Tset"] == t)]
                if sub.empty:
                    continue
                cm = cond_table(sub).sort_values("ln_gd")
                lg = np.linspace(sub["ln_gd"].min(), sub["ln_gd"].max(), 120)
                Xg = pd.DataFrame({"ln_gd": lg, "ln_gd_cm": lg, "ln_H": np.log(h), "Tset": float(t), "ln_Tff": np.interp(lg, cm["ln_gd"], cm["ln_Tff"]), "ln_Tff_cm": np.interp(lg, cm["ln_gd"], cm["ln_Tff"])})
                mu, sd, _, _ = predict_variant(fit, Xg)
                ax.plot(np.exp(lg), np.exp(mu), color=col, lw=1.4)
                ax.fill_between(np.exp(lg), np.exp(mu - 1.96 * sd), np.exp(mu + 1.96 * sd), color=col, alpha=0.12)
                is_te = np.array([c in te_cond for c in cm["cond"]])
                ax.scatter(np.exp(cm["ln_gd"][~is_te]), np.exp(cm["ln_eta"][~is_te]), s=14, color=col, alpha=0.8, zorder=3)
                ax.scatter(np.exp(cm["ln_gd"][is_te]), np.exp(cm["ln_eta"][is_te]), s=26, facecolors="none", edgecolors=col, marker="s", linewidths=1.2, zorder=3)
                sens = cm["is_sens"].to_numpy().astype(bool)
                ax.scatter(np.exp(cm["ln_gd"][sens]), np.exp(cm["ln_eta"][sens]), s=45, color="k", marker="x", zorder=4)
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.grid(alpha=0.3, which="both")
            if j == 0:
                ax.set_ylabel(f"厚み {h} mm\n" + r"$\eta$ [Pa·s]")
            if i == 0:
                v = fit["v"]
                ax.set_title(STRUCT_JA[name] + ("" if v["kind"] == "free" else f"\n(E_h, E_v) = ({v['Eh']:.1f}, {v['Ev']:.1f})"), fontsize=9)
            if i == 3:
                ax.set_xlabel(r"$\dot{\gamma}$ [1/s]")
    handles = [plt.Line2D([], [], color=c, label=f"{t} ℃") for t, c in T_COL.items()]
    handles += [plt.Line2D([], [], color="gray", marker="o", ls="", label="学習の条件平均"), plt.Line2D([], [], color="gray", marker="s", mfc="none", ls="", label="検証の条件平均"),
                plt.Line2D([], [], color="k", marker="x", ls="", label="感度確認の条件")]
    axes[0, 0].legend(handles=handles, fontsize=7, loc="upper right")
    fig.suptitle(f"{MATS[mat]}：設定温度ごとの予測曲線（5data で学習。線：予測、帯：95% 予測区間）", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    fig.savefig(os.path.join(OUT_DIR, f"fig4{'a' if mat == 'PP' else 'b'}_curves_{mat}.png"), dpi=FIG_DPI)
    plt.show()

In [ ]:
# セル13：図5 評価指標の比較、図6 設定温度の効果
PANELS = [("S_V", "低速側", "S_V 低速側"), ("S_V", "高速側", "S_V 高速側"), ("S_T", "全体", f"S_T（{TEMP_HOLDOUT} ℃）"), ("S_T3", "全体", f"S_T3（3data の {TEMP_HOLDOUT} ℃）")]
COLORS = ["tab:blue", "tab:red", "tab:purple", "tab:brown"]
fig, axes = plt.subplots(2, len(MATS), figsize=(16, 9))
for j, mat in enumerate(MATS):
    for i, (metric, ylab) in enumerate([("RMSE", r"RMSE（$\ln\eta$）"), ("NLPD", "NLPD（小さいほど良い）")]):
        ax = axes[i, j]
        w = 0.8 / len(PANELS)
        for k, (split, sub, lab) in enumerate(PANELS):
            ax.bar(np.arange(len(VARS)) + (k - (len(PANELS) - 1) / 2) * w, [m5(mat, split, v, sub, metric) for v in VARS], w, color=COLORS[k], label=lab)
        ax.set_xticks(np.arange(len(VARS)))
        ax.set_xticklabels(VARS)
        ax.set_title(f"{MATS[mat]}：{ylab}（入力の " + r"$\dot{\gamma}$" + " は条件平均）")
        ax.grid(alpha=0.3, axis="y")
        if metric == "RMSE":
            ax.set_yscale("log")
        if i == 0 and j == 0:
            ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig5_metrics.png"), dpi=FIG_DPI)
plt.show()

fig, axes = plt.subplots(1, len(MATS), figsize=(15, 5.2), sharey=True)
MK = {"C": "s", "N": "x", "H": "o", "H40": "^", "A": "v", "Hz": "D", "B": "P", "F": "X"}
for ax, mat in zip(axes, MATS):
    o = obs5[obs5["material"] == MATS[mat]]
    ax.plot(o["H"], o["observed"], "k-o", lw=2.5, ms=9, label=r"実測（同じ $\dot{\gamma}$）", zorder=10)
    for name in VARS:
        e = eff5[(eff5["split"] == "S_V") & (eff5["material"] == MATS[mat]) & (eff5["variant"] == name)]
        ax.plot(e["H"], e["ratio_190_230"], marker=MK[name], ls="--", alpha=0.85, color=STRUCT_COL[name], label=name)
    ax.axhline(1.0, color="gray", lw=0.8)
    ax.set_xlabel("厚み [mm]")
    ax.set_title(f"{MATS[mat]}：190 ℃ と 230 ℃ の粘度の比（Tset だけを変えた予測、S_V の学習）")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("粘度の比（190 ℃ ÷ 230 ℃）")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig6_temperature_effect.png"), dpi=FIG_DPI)
plt.show()

## 10. 判定のまとめ

OK は想定どおり、★ は想定と違う、または注意が要る点。基準は、判定の文の中に書いてある。

In [ ]:
# セル14：判定
def bE(mat, sname, st, col="E_best"):
    return float(e_of(mat, sname, st)[col])


print("【判定】")
for mat in MATS:
    n = MATS[mat]
    eH = bE(mat, "S_V", "H")
    dA, dH = float(e_of(mat, "S_V", "A")["dLML_vs_E0"]), float(e_of(mat, "S_V", "H")["dLML_vs_E0"])
    judge(dA - dH >= 20.0, f"{n}：A の LML の改善（温度なし E = 0 との差）{dA:.1f} が、H の改善 {dH:.1f} より 20 以上大きい（H は、E = −20〜12.5 でほぼ平坦で、E がデータで決まらない。10-12 の試算でも、H の最良は 0 付近か負）")
    judge(eH >= 20.0, f"{n}：H の最良の E = {eH:.1f} kJ/mol が、文献値の目安（40）に近い（20 以上）なら、§5 の構造（H）はデータで支持される")
    dh = float(e_of(mat, "S_V", "H")["LML_best"] - e_of(mat, "S_V", "H")["LML_at_E40"])
    judge(dh < 10.0, f"{n}：H の LML は、最良の E に比べて、E = 40 では {dh:.1f} 下がる（10 未満なら、40 でも許される。10 以上なら、40 はデータと合わない）")
    lA, lH, lB, lZ = (float(e_of(mat, "S_V", s)["LML_best"]) for s in ["A", "H", "B", "Hz"])
    judge(lA - lH >= 2.0, f"{n}：A（縦だけ）の LML が H より 2 以上大きい（A {lA:.1f}、H {lH:.1f}）なら、縦だけのシフトのほうがデータに合う")
    judge(lB - max(lA, lH, lZ) >= 2.0, f"{n}：B（横・縦別々）の LML が A・H・Hz のどれより 2 以上大きい（B {lB:.1f}、最大の他 {max(lA, lH, lZ):.1f}）なら、横と縦の両方が必要")
    d0 = float(e_of(mat, "S_V", "A")["dLML_vs_E0"])
    judge(d0 >= 2.0, f"{n}：A の最良の E_v = {bE(mat, 'S_V', 'A'):.1f} kJ/mol（95% 区間 {bE(mat, 'S_V', 'A', 'E_95_lo'):.1f}〜{bE(mat, 'S_V', 'A', 'E_95_hi'):.1f}）。温度なし（E = 0）より LML が 2 以上大きい（差 {d0:.1f}）なら、温度の効果がデータで支持される")
    for st in ["H", "A"]:
        es = [bE(mat, s, st) for s in ["S_V", "S_T", "ALL"]]
        judge(max(es) - min(es) <= 5.0, f"{n}：{st} の最良の E が、学習データ（S_V・S_T・ALL）を変えても 5 kJ/mol 以内（{', '.join(f'{x:.1f}' for x in es)}）")
    edge = [st for st in ["H", "A", "Hz", "F"] if bool(e_of(mat, "S_V", st)["at_grid_edge"])]
    judge(not edge, f"{n}：最良の E が格子の端に張り付いている構造: {edge or 'なし'}（張り付いたら、格子の範囲を広げる）")
    for split, sub, lab in [("S_V", "低速側", "S_V 低速側"), ("S_V", "高速側", "S_V 高速側"), ("S_T", "全体", "S_T")]:
        c = m5(mat, split, "C", sub)
        best = min(["A", "B", "H"], key=lambda s: m5(mat, split, s, sub))
        judge(m5(mat, split, best, sub) <= 1.1 * c, f"{n} {lab}：構造ありの最良（{best}、RMSE {m5(mat, split, best, sub):.3f}）が、通常の GPR（C、{c:.3f}）の 1.1 倍以内（同等以上なら、構造がデータで支持される）")
    r = eff5[(eff5["split"] == "S_V") & (eff5["material"] == n)]
    o = obs5[obs5["material"] == n]["observed"]
    for st in ["A", "B", "H"]:
        e_ = r[r["variant"] == st]["ratio_190_230"]
        judge(bool((e_ > 1.0).all()), f"{n} {st}：Tset だけを 190 → 230 ℃ にした予測の粘度比が、全厚みで 1 を超える（範囲 {e_.min():.2f}〜{e_.max():.2f}、実測 {o.min():.2f}〜{o.max():.2f}）")
print("\n保存したファイル:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  ", os.path.normpath(os.path.join(OUT_DIR, f)))

## 11. 高解像度のグラフ（報告書・発表資料用）

せん断速度 ― 粘度の学習結果グラフを、GPR6月ノートのセル9-2 と同じ見せ方で、**600 dpi（15 × 10 インチ）**で 1 枚ずつ保存する。線は予測の平均、帯は 95% 区間、塗りつぶしの点は学習に使った条件、白抜きの点は学習に使わなかった条件（検証）、× は問題データ一覧の「感度確認」の条件。

- **構造**：既定は、本命（`MAIN_STRUCT`）と対案（`ALT_STRUCT`）。`HR_STRUCTS` で、H・A・Hz・B・N・H40・F・C のどれでも選べる。
- **レイアウト**：`"thickness"`＝厚みごとに 1 枚（3 つの設定温度の曲線）、`"temperature"`＝設定温度ごとに 1 枚（4 つの厚みの曲線。6月のノートと同じ）。
- **変えられるもの**（セルの先頭の設定エリア）：描くモデル・材料・学習の種類、帯（予測区間／信頼区間）、点（条件平均／ショットごと）、軸の範囲、予測線の延長、図の大きさ・解像度・文字・マーカーの大きさ、タイトルと軸名の表示、保存形式（png／pdf／svg）。
- 全体を一度実行して、学習が終わった後なら、このセルだけを何度でも実行し直せる。

In [ ]:
# セル15：高解像度のグラフ（せん断速度 ― 粘度。本命と対案の予測。GPR6月ノートのセル9-2 にならう）
# ==========================================
# ▼▼ 設定エリア（グラフの見た目・文字サイズ。このセルの設定だけを直して、このセルだけを実行し直せる） ▼▼
# ==========================================
HR_STRUCTS = [MAIN_STRUCT, ALT_STRUCT]   # 描く構造（既定は、本命と対案。H・A・Hz・B・N・H40・F・C のどれでも。記号の意味は、ノート冒頭の表）
HR_SPLIT = "S_V"                   # どの学習のモデルか："S_V"（5data で学習）／"S_T"（190・230 ℃ で学習）／"ALL"（5data＋3data で学習し直す。点は全部が学習の点）
HR_MATERIALS = ["PP", "MIX"]       # 描く材料（PP：PP、MIX：粉砕材混合 PP）
HR_LAYOUTS = ["thickness", "temperature"]   # "thickness"：厚みごとに 1 枚（3 つの設定温度）／"temperature"：設定温度ごとに 1 枚（4 つの厚み。6月のノートと同じ）
HR_BAND = "predict"                # 帯："predict"＝95% 予測区間（ショット間のばらつき込み）／"latent"＝95% 信頼区間（平均の不確かさだけ）／None＝帯なし
HR_POINTS = "cond"                 # 点："cond"＝条件平均／"shot"＝ショットごと
HR_SHOW_TEST = True                # 学習に使わなかった条件（検証）を、白抜きで重ねるか
HR_MARK_SENS = True                # 問題データ一覧の「感度確認」の条件を、× で重ねるか
HR_X_RANGE = None                  # 予測線の範囲 [1/s]。None なら、各線のデータの範囲を HR_EXTEND 倍に延長。(30, 50000) のように指定すると、全部の線をその範囲に延長（外挿の確認用）
HR_EXTEND = 1.3
HR_XLIM = {"thickness": None, "temperature": (30, 50000)}   # 横軸（せん断速度 [1/s]）の表示範囲。(下限, 上限)＝固定（10 の累乗の目盛り。6月と同じ）／None＝その図のデータに合わせる（1・2・5 の目盛り）。
HR_YLIM = {"thickness": None, "temperature": (0.5, 5000)}   # 縦軸（粘度 [Pa·s]）の表示範囲。同じ書き方。レイアウトごとに変えられる（全部を同じ範囲にするなら、(30, 50000) のように書く）
HR_FIGSIZE = (15, 10)              # 図の大きさ（幅, 高さ）[インチ]
HR_DPI = 600                       # 解像度
HR_FONT = {"axis": 24, "legend": 24, "title": 24, "tick": 28}   # 軸名・凡例・タイトル・目盛り数字の文字サイズ
HR_MARKER = 200                    # 点の大きさ（条件平均のとき。面積）
HR_MARKER_SHOT = 70                # 点の大きさ（ショットごとのとき）
HR_LINE = 3.0                      # 予測線の太さ
HR_SHOW_TITLE = True               # タイトルを付けるか
HR_SHOW_XLABEL = False             # 軸名を付けるか（6月のノートと同じく、既定は付けない。資料に貼るとき、別に付ける場合のため）
HR_SHOW_YLABEL = False
HR_FORMATS = ["png"]               # 保存形式。"png"・"pdf"・"svg" から選ぶ（pdf・svg は、拡大しても粗くならない）
HR_SHOW = False                    # True なら、画面にも表示する（重い）
# ==========================================
HR_KW = dict(layouts=HR_LAYOUTS, band=HR_BAND, points=HR_POINTS, show_test=HR_SHOW_TEST, mark_sens=HR_MARK_SENS, x_range=HR_X_RANGE, extend=HR_EXTEND,
             xlim=HR_XLIM, ylim=HR_YLIM, figsize=HR_FIGSIZE, dpi=HR_DPI, font=HR_FONT, marker=HR_MARKER, marker_shot=HR_MARKER_SHOT, line=HR_LINE,
             show_title=HR_SHOW_TITLE, show_xlabel=HR_SHOW_XLABEL, show_ylabel=HR_SHOW_YLABEL, formats=HR_FORMATS, show=HR_SHOW)
HR_DIR = os.path.join(OUT_DIR, "highres")

HR_SPLIT_JA = {"S_V": "5data で学習", "S_T": "190・230 ℃ で学習", "ALL": "5data＋3data で学習"}


def hr_fit5(mat, name):
    """描く構造の学習済み予測器（入力の γ̇ は条件平均）。S_V・S_T は上で学習したもの、ALL は 5data＋3data で学習し直す（E は、ALL のプロファイルの最良の値）"""
    if HR_SPLIT == "ALL":
        return fit_variant(variant_defs(mat, "ALL")[name], pool_of(mat, MAIN_DATA), "cm", N_RESTARTS_FINAL)
    return FIT5[(mat, HR_SPLIT, name, "cm")]


hr_saved = []
for mat in HR_MATERIALS:
    for name in HR_STRUCTS:
        t0 = time.time()
        fit = hr_fit5(mat, name)
        v = fit["v"]
        label = STRUCT_JA[name] + ("" if v["kind"] == "free" else f"　E_h = {v['Eh']:.1f}、E_v = {v['Ev']:.1f} kJ/mol") + f"（{HR_SPLIT_JA[HR_SPLIT]}）"
        hr_saved += hr_flow(lambda Xg, f=fit: predict_variant(f, Xg)[:3], mat, MAIN_DATA, hr_split_conds(mat, MAIN_DATA, HR_SPLIT, TEMP_HOLDOUT),
                            label, f"hr_{mat}_{name}_{HR_SPLIT}", HR_DIR, **HR_KW)
        print(f"{MATS[mat]} {name}：保存したファイル 累計 {len(hr_saved)} 個（{time.time() - t0:.0f} 秒）")
judge(len(hr_saved) > 0, f"高解像度のグラフを {len(hr_saved)} 個保存した（{os.path.normpath(HR_DIR)}）。形式 {HR_FORMATS}、{HR_DPI} dpi")